In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

import os


In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'II_USA NYSBD' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

# writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running II_USA NYSBD Web Scraping Tool v.1.2


In [3]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()





In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict= {

    	  'II_USA NYSBD 1': ['Bank'],

		  'II_USA NYSBD 2': ['Credit Union'],

		  'II_USA NYSBD 3': 'https://myportal.dfs.ny.gov/web/guest-applications/ins.-company-search',

		  'II_USA NYSBD 5': ['Foreign Bank Agency'],

		  'II_USA NYSBD 6': ['Foreign Bank Branch'],

		  'II_USA NYSBD 7': ['Foreign Bank Representative Office'],

		  #'II_USA NYSBD 8': ['Holding Co - One Bank'],

		  'II_USA NYSBD 9': ['Investment Company'],

		  'II_USA NYSBD 10': ['Licensed Lender'],

		  'II_USA NYSBD 11': ['Money Transmitter'],

		  'II_USA NYSBD 12': ['Mortgage Banker (Exempt)', 'Mortgage Banker Non-Profit (Exempt)', 'Mortgage Broker (Exempt)', 'Mortgage Broker Non-Profit (Exempt)', 'Mortgage Loan Servicer (Exempt)',

		  					  'Mortgage Banker (Exempt - UW)','Mortgage Banker', 'Mortgage Broker', 'Mortgage Loan Servicer'],

		  'II_USA NYSBD 13': ['NYS Regulated Corporation'],

		  'II_USA NYSBD 14': ['Premium Finance Agency'],

		  'II_USA NYSBD 15': ['Private Banker'],

		#   'II_USA NYSBD 16': ['Safe Deposit Company'],

		  'II_USA NYSBD 17': ['Sales Finance Company'],

		  'II_USA NYSBD 18': ['Savings Bank'],

		  'II_USA NYSBD 19': ['Trust Fund (Common)'],

		  #'II_USA NYSBD 20': ['Bank Holding Company'],

		  'II_USA NYSBD 21': ['Mutual Holding Company'],

		  'II_USA NYSBD 22': ['Trust Company']

          }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



states= {'':'', 'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'DC': 'District of Columbia', 'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming', 'AS': 'American Samoa', 'GU': 'Guam', 'MP': 'Northern Mariana Islands', 'PR': 'Puerto Rico', 'VI': 'US. Virgin Islands', 'UM': 'US. Minor Outlying Islands'}

processdate = now.strftime('%Y-%m-%d')





In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



In [6]:


# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

	print(f"[INFO] : Working {k+1}/{len(regdict)} ({reg}) ")

	datadict={'NAIC_Num': [], 'Company_Name': [], 'Org': [], 'Domicile': [], 'GNu_GNa': [], 'FId': [], 'Website': []}

	

	if type(regdict[reg]) is list:

		driver.get('https://myportal.dfs.ny.gov/web/guest-applications/who-we-supervise?null=')

		sleep(3)

		for re in range(len(regdict[reg])):

			xpath=f'//*/ul[@id="_BIsearch2_WAR_Blsearch2_:instype_items"]/li[contains(text(),"{regdict[reg][re]}")]'

			instype_input = driver.find_element(By.XPATH, '//*[@id="_BIsearch2_WAR_Blsearch2_:instype_label"]')

			driver.execute_script("arguments[0].click();", instype_input)

			sleep(0.5)



			for times in range(10):

				try:

					instype_items = driver.find_element(By.XPATH, xpath)

					driver.execute_script("arguments[0].click();", instype_items)

					break

				except:

					print(f"[ERROR] : - Try {times}/10 to select Institution '{regdict[reg][re]}' ")

					sleep(1)

			else:

				Exception(f'[ERROR] : - Failed to select Type of Institution: "{regdict[reg][re]}", \nrError: ')

				

			buttons=driver.find_elements(By.XPATH, "//button")

			for button in buttons:

				if 'SEARCH' in button.text.upper():

					driver.execute_script("arguments[0].click();", button)

					break

			else:

				Exception(f'[ERROR] : - Failed to click in xpath: "{xpath}", \nrError: ')

			

			sleep(1)

			for times in range(10):

				if 'Download Search Results' in driver.page_source:

					break

				else:

					sleep(1)

			csvurl=driver.find_element(By.XPATH, "//a[text()[contains(.,'CSV')]]")

			driver.execute_script("arguments[0].click();", csvurl)

			sleep(0.2)#to avoid emptylist bypass in line 99, additional check could be done.

			file =  check_dowload_files(tempfolder, "csv",30 )

			filePath = os.path.join(tempfolder, file)

					

			if re==0:#single element list

				df=pd.read_csv(filePath, dtype=str)

			else:#for mortgage 

				tempdf=pd.read_csv(filePath, dtype=str)

				tempdf=tempdf.drop(tempdf.index[0])

				# df=df.append(tempdf)
				df = pd.concat([df, tempdf], ignore_index=True)


				df=df.reset_index(drop=True)

			

			for rem in os.listdir(tempfolder):

				os.remove(os.path.join(tempfolder, rem))

				

			df=df.fillna('')

			print(f"[INFO] : - DataFrame '{file}' | containe = {df.shape}")

			reglenght=range(len(df['PRIMARY NAME'].to_list()))

			address=[addr_.split(',') for addr_ in df['MAIN ADDRESS'].to_list()]

			for addr in address:

				if len(address)>1:

					sqldict['Address_1'].append(','.join(addr[:-1]).strip())

					if addr[-1][:2].isalpha() and ' '==addr[-1]:

						zipcode=addr[-1].strip()

						sqldict['Zip'].append(addr[-1].strip())

						sqldict['City'].append(states[zipcode[:2]])

					else:

						sqldict['Zip'].append('')

						sqldict['City'].append('')

				else:

					sqldict['Address_1'].append(addr[0].strip())

			sqldict['Name'].extend(df['PRIMARY NAME'].to_list())

			sqldict['Cntry'].extend(['US' for rang in reglenght])

			sqldict['RegCtry'].extend(['II_USA' for rang in reglenght])#list(map(lambda x: 'II_USA', reglenght)))

			sqldict['RegCode'].extend(['NYSBD' for rang in reglenght])#(list(map(lambda x: 'NYSBD', reglenght)))

			sqldict['ListCode'].extend([reg.split(' ')[-1] for rang in reglenght])#(list(map(lambda x: reg.split(' ')[-1], reglenght)))

			sqldict['Typology'].extend([regdict[reg][re] for rang in reglenght])#(list(map(lambda x: regdict[reg][re], reglenght)))

			sqldict['RegulationType'].extend(['Supervised' for rang in reglenght])#(list(map(lambda x: 'Supervised', reglenght)))

			sqldict['ListProcessDate'].extend([processdate for rang in reglenght])#(list(map(lambda x: processdate, reglenght)))

			for key in sqldict.keys():

				while len(sqldict[key])<len(sqldict['Name']):

					sqldict[key].append('')



	else:#for insurance regcode:

		driver.get(regdict[reg])

		sleep(3)

		driver.switch_to.frame(driver.find_element(By.TAG_NAME, 'iframe'))

		while True:

			try:

				inputs=driver.find_elements(By.XPATH, '//input[@value="Search"]')

				driver.execute_script("arguments[0].click();", inputs[4])

				break

			except:

				print('[INFO] : Waiting 5 seconds before retry.')

				sleep(5)

		sleep(10)

		source=driver.page_source

		soup=BeautifulSoup(source, 'html.parser')

		table=soup.find('table', {'cellpadding':'3'})

		table=table.find('tbody')

		trs=table.find_all('tr')[1:]

		print(f"[INFO] : - table containe = {len(trs)} Companies")

		for tr in trs:

			tds=tr.find_all('td')

			sqldict['Name'].append(tds[1].text.strip())

			sqldict['InternalID_1_type'].append('NAIC Number')

			sqldict['InternalID_1'].append(tds[0].text.strip())

			sqldict['Website'].append(tds[6].text.strip())

			sqldict['InternalID_2_type'].append('FID')

			sqldict['InternalID_2'].append(tds[5].text.strip())

			sqldict['City'].append(tds[3].text.strip())

			sqldict['Cntry'].append('US')#verify

			sqldict['RegCtry'].append('US')

			sqldict['RegCode'].append('NYSBD')

			sqldict['ListCode'].append(reg.split(' ')[2])

			sqldict['Typology'].append('Insurance Company')

			sqldict['RegulationType'].append('Regulated')

			sqldict['ListProcessDate'].append(processdate)

			#datadict['Org'].append(tds[2].text.strip())

			#datadict['GNu_GNa'].append(tds[4].text.strip())

			sqldict = bourange_same_length_array(sqldict)




[INFO] : Working 1/19 (II_USA NYSBD 1) 
[INFO] : - Download csv file ... (wait 0/20 s)
[INFO] : - csv file = ['searchList.csv'])
[INFO] : - DataFrame 'searchList.csv' | containe = (29, 6)
[INFO] : Working 2/19 (II_USA NYSBD 2) 
[INFO] : - csv file = ['searchList.csv'])
[INFO] : - DataFrame 'searchList.csv' | containe = (9, 6)
[INFO] : Working 3/19 (II_USA NYSBD 3) 
[INFO] : - table containe = 2069 Companies
[INFO] : Working 4/19 (II_USA NYSBD 5) 
[INFO] : - Download csv file ... (wait 0/20 s)
[INFO] : - csv file = ['searchList.csv'])
[INFO] : - DataFrame 'searchList.csv' | containe = (10, 6)
[INFO] : Working 5/19 (II_USA NYSBD 6) 
[INFO] : - csv file = ['searchList.csv'])
[INFO] : - DataFrame 'searchList.csv' | containe = (67, 6)
[INFO] : Working 6/19 (II_USA NYSBD 7) 
[INFO] : - Download csv file ... (wait 0/20 s)
[INFO] : - csv file = ['searchList.csv'])
[INFO] : - DataFrame 'searchList.csv' | containe = (31, 6)
[INFO] : Working 7/19 (II_USA NYSBD 9) 
[INFO] : - Download csv file ...

In [7]:



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

# writer.save()

# writer.close()

driver.quit()

sleep(3)
    

NameError: name 'writer' is not defined

In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Bank,,Adirondack Bank,,,,,...,,,,,,,,,,
1,,,,Bank,,Alden State Bank,,,,,...,,,,,,,,,,
2,,,,Bank,,Alma Bank,,,,,...,,,,,,,,,,
3,,,,Bank,,Alpine Capital Bank,,,,,...,,,,,,,,,,
4,,,,Bank,,Amerasia Bank,,,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436580,,,,Trust Company,,NYDIG Trust Company LLC,,,,,...,,,,,,,,,,
436581,,,,Trust Company,,"PayPal Digital, Inc.",,,,,...,,,,,,,,,,
436582,,,,Trust Company,,"Standard Custody & Trust Company, LLC",,,,,...,,,,,,,,,,
436583,,,,Trust Company,,"Vitesse Trust Company, LLC",,,,,...,,,,,,,,,,
